## Preprocessing related to paper

### Data Ingestion

Copy all the relevant files from the /ETL/data folder after running the ETL pipeline.

These correspond to the following folders:
- annual_electricity_demand
- electricity_demand
- gdp
- temperature

Note: Skip parts of the code for variables you do not want to include in the final dataset.

#### Imports

In [ ]:
import os

import pandas
import xarray
from tqdm import tqdm

#### Annual Electricity Demand

In [ ]:
# Specify the folder and file type
electricity_annual_demand_folder = "./data/annual_electricity_demand/"
electricity_annual_demand_files = [
    file_name
    for file_name in os.listdir(electricity_annual_demand_folder)
    if file_name.endswith(".parquet")
]

In [ ]:
# Load all the files into one DataFrame
df_annual_demand = pandas.DataFrame()

for file_name in tqdm(electricity_annual_demand_files):
    df_current = pandas.read_parquet(
        electricity_annual_demand_folder + file_name
    )

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = file_name.split(".")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_annual_demand = pandas.concat(
        [df_annual_demand, df_current], ignore_index=True
    )

In [ ]:
print(df_annual_demand.shape)
df_annual_demand.head()

#### Electricity Demand

In [ ]:
# Specify the folder and file type
electricity_demand_folder = "./data/electricity_demand/"
demand_files = [
    file_name
    for file_name in os.listdir(electricity_demand_folder)
    if file_name.endswith(".parquet")
]

In [ ]:
# Load all the files into one DataFrame
df_demand = pandas.DataFrame()

for file_name in tqdm(demand_files):
    df_current = pandas.read_parquet(electricity_demand_folder + file_name)

    df_current["Load (MW)"] = df_current["Load (MW)"].astype(float)

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = str.join("_", file_name.split("_")[:-1])

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_demand = pandas.concat([df_demand, df_current], ignore_index=True)

In [ ]:
print(df_demand.shape)
df_demand.head()

#### GDP

In [ ]:
# Specify the folder and file type
gdp_folder = "./data/gdp/"
gdp_files = [
    file_name
    for file_name in os.listdir(gdp_folder)
    if file_name.endswith(".nc")
]

In [ ]:
# Load all the files into one DataFrame
df_gdp_data = pandas.DataFrame()

for file_name in tqdm(gdp_files):
    # Extract region code from filename
    region_code = file_name.split("_0.25_deg_")[0]
    year = int(file_name.split("_0.25_deg_")[-1].replace(".nc", ""))

    # Open the NetCDF file
    gdp_data = xarray.open_dataset(gdp_folder + file_name)

    # Extract GDP value, sum over the entire area
    gdp_value = float(gdp_data.gdp.to_numpy().sum())

    # Create a DataFrame for this file
    df_current = pandas.DataFrame(
        {"year": [year], "GDP": [gdp_value], "region_code": [region_code]}
    )

    # Extract country code (assuming it's the first part of region_code)
    country_code = region_code.split("_")[0]
    df_current["country_code"] = country_code

    # Append to the main DataFrame
    df_gdp_data = pandas.concat([df_gdp_data, df_current], ignore_index=True)

In [ ]:
print(df_gdp_data.shape)
df_gdp_data.head()

#### Temperature (Weather)

In [ ]:
# Specify the folder and file type
temperature_folder = "./data/temperature/"
temperature_files = [
    file_name
    for file_name in os.listdir(temperature_folder)
    if file_name.endswith(".parquet")
]

In [ ]:
# Load all the files into one DataFrame
df_all_temperature = pandas.DataFrame()

for file_name in tqdm(temperature_files):
    df_current = pandas.read_parquet(temperature_folder + file_name)

    # Add a column for the region name
    df_current["region_code"] = file_name.split("_temp")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_all_temperature = pandas.concat(
        [df_all_temperature, df_current], ignore_index=True
    )

In [ ]:
print(df_all_temperature.shape)
df_all_temperature.head()

### Merge data into one larger dataset

Keep in mind to skip any code blocks corresponding to variables not loaded in the above code.

The order of merging is based on empirical evidence based on the feature importance after training a XGBoost model.
Feel free to adjust it to your needs.

#### Merge Temperature and Demand Data

In [ ]:
df_all_temperature = df_all_temperature.sort_values(by=["Time (UTC)"])
df_demand = df_demand.sort_values(by=["Time (UTC)"])

In [ ]:
# Merge the demand data
total_dataset = pandas.merge(
    df_all_temperature, df_demand, on=["Time (UTC)", "region_code"]
)

In [ ]:
print(total_dataset.shape)
total_dataset.head()

#### Merge Annual Electricity Demand

In [ ]:
# Run to add annual demand data to the dataset
total_dataset = pandas.merge(
    total_dataset, df_annual_demand, on=["Time (UTC)", "region_code"]
)
# Scale the yearly demand from TW to MW
total_dataset["year_electricity_demand_mw"] = (
    total_dataset["Annual electricity demand (TWh)"] * 1000000
)
total_dataset = total_dataset.drop(columns=["Annual electricity demand (TWh)"])

In [ ]:
print(total_dataset.shape)
total_dataset.head()

#### Merge GDP

In [ ]:
total_dataset = pandas.merge(
    total_dataset,
    df_gdp_data.drop(columns=["country_code"]),
    left_on=["Local year", "region_code"],
    right_on=["year", "region_code"],
)
total_dataset = total_dataset.drop(columns=["year"])

In [ ]:
print(total_dataset.shape)
total_dataset.head()

#### Renaming to simplify column names

In [ ]:
total_dataset = total_dataset.rename(
    columns={
        "Time (UTC)": "time_utc",
        "Local hour of the day": "local_hour",
        "Local weekend indicator": "is_weekend",
        "Local month of the year": "local_month",
        "Local year": "local_year",
        "Temperature - Top 1 (K)": "year_temp_top1",
        "Temperature - Top 3 (K)": "year_temp_top3",
        "Monthly average temperature - Top 1 (K)": "monthly_temp_avg_top1",
        "Monthly average temperature rank - Top 1": "monthly_temp_avg_rank_top1",
        "Annual average temperature - Top 1 (K)": "year_temp_avg_top1",
        "5 percentile temperature - Top 1 (K)": "year_temp_percentile_5",
        "95 percentile temperature - Top 1 (K)": "year_temp_percentile_95",
        "Annual electricity demand (TWh)": "year_electricity_demand",
        "Annual electricity demand per capita (MWh)": "year_electricity_demand_per_capita_mwh",
        "Load (MW)": "load_mw",
        "GDP": "year_gdp",
    }
)
print(total_dataset.shape)
total_dataset.head()

#### Post-processing

##### Duplicate and NaN removal

In [ ]:
# Remove duplicates
row_count = len(total_dataset)
print("Before removing duplicates:", row_count)
total_dataset = total_dataset.drop_duplicates(
    subset=[col for col in total_dataset.columns if col != "load_mw"]
)
print("Without duplicates: ", len(total_dataset))
print("Difference", row_count - len(total_dataset))

In [ ]:
# Remove NaN values
row_count = len(total_dataset)
print("Before removing NaN values:", row_count)
total_dataset = total_dataset.dropna()
print("Without duplicates: ", len(total_dataset))
print("Difference", row_count - len(total_dataset))

##### Calculate the percentage that each hour represents of the yearly load

In [ ]:
for name, group in total_dataset.groupby(["region_code", "local_year"]):
    yearly_load = group["load_mw"].sum()
    amount_of_hours_tracked = len(group["load_mw"])
    # Calculate the amount of hours in the specified year,
    # accounting for leap years
    amount_of_hours_in_year = (
        len(
            pandas.date_range(start=f"{name[1]}-01-01", end=f"{name[1]}-12-31")
        )
        * 24
    )

    # Get the percentage that load_mw represents of the yearly load
    load_mw_percentage = group["load_mw"] / yearly_load

    # Adjust the percentages to account for missing hours
    total_dataset.loc[group.index, "load_mw_percentage"] = (
        load_mw_percentage
        * (amount_of_hours_tracked / amount_of_hours_in_year)
    )

### Save the dataset

In [ ]:
total_dataset.to_parquet("./data/total_dataset.parquet", engine="pyarrow")

#### Details and visuals of the total dataset

In [ ]:
# Investigate the distribution of available hours per region and year
list_amount_hours_region = []
for name, group in total_dataset.groupby(["region_code", "local_year"]):
    list_amount_hours_region.append([name[0], name[1], len(group)])

df_amount_hours_region = pandas.DataFrame(
    list_amount_hours_region,
    columns=["region_code", "local_year", "count_available_hours"],
)

df_amount_hours_region["count_available_hours"].hist(bins=10)